# Multi-Head Attention

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

One head can only learn one *kind* of relationship. Multi-head attention runs $h$ independent attentions in parallel on lower-dimensional projections of the input, then concatenates and re-projects. Each head can specialise (syntax, coreference, position, etc.).


## Mathematical Formulation

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\,W^O$$

$$\text{head}_i = \text{Attention}(XW^Q_i,\, XW^K_i,\, XW^V_i)$$

Each head projects $X \in \mathbb{R}^{n \times d}$ down to $d/h$ dimensions, performs scaled dot-product attention, and the outputs are concatenated to recover $d$.


## Implementation


In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads = d_model, n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, N, _ = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)                      # (B, N, H, D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v)) # (B, H, N, D)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))
        attn = self.drop(F.softmax(scores, dim=-1))
        ctx = (attn @ v).transpose(1, 2).reshape(B, N, self.d_model)
        return self.out(ctx), attn


## Experiment


In [ ]:
mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
y, attn = mha(x)
print('output:', y.shape)
print('per-head attention:', attn.shape)  # (B, H, N, N)


In [ ]:
# Quick visualisation of head 0 attention for batch 0
import matplotlib.pyplot as plt
plt.imshow(attn[0, 0].detach().numpy(), cmap='viridis')
plt.colorbar(); plt.title('head 0'); plt.xlabel('key'); plt.ylabel('query')
plt.show()


## Discussion

- The QKV projection is fused into a single linear for efficiency (3× wider).
- Each head sees the full sequence but only $d/h$ channels — that's where specialisation comes from.
- Dropout sits *inside* the softmax weights, not on the output; this matches the original Transformer paper.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
